# 03 — Prepare & Export

Build "Without abortion" and "With abortion" comparison tables.

**Steps:**
1. Load mortality data and abortion estimates
2. Create comparison tables (without/with abortion)
3. Export to CSV, Excel, Parquet + codebook


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')


## Step 1: Load Data


In [ ]:
mort_national = run_sql('SELECT * FROM mortality_national ORDER BY deaths DESC', con)
mort_female = run_sql('SELECT * FROM mortality_female ORDER BY deaths DESC', con)
mort_repro = run_sql('SELECT * FROM mortality_female_repro ORDER BY deaths DESC', con)
df_abortions = run_sql('SELECT * FROM abortions', con)

print('Data loaded:')
print(f'  National: {len(mort_national)} causes')
print(f'  Female: {len(mort_female)} causes')
print(f'  Female 15-44: {len(mort_repro)} causes')


## Step 2: Extract Abortion Totals


In [ ]:
abortion_national = df_abortions[df_abortions['measure'] == 'national_total']['value'].iloc[0]
abortion_repro = df_abortions[df_abortions['measure'] == 'repro_age_total']['value'].iloc[0]
abortion_female = abortion_national

print(f'Abortion totals (2024):')
print(f'  National: {abortion_national:,.0f}')
print(f'  Female (all ages): {abortion_female:,.0f}')
print(f'  Female 15-44: {abortion_repro:,.0f}')


## Step 3: Calculate Adjusted Populations


In [ ]:
mort_national['population_adjusted'] = mort_national['population'] + abortion_national
mort_female['population_adjusted'] = mort_female['population'] + abortion_female
mort_repro['population_adjusted'] = mort_repro['population'] + abortion_repro

print('Adjusted populations calculated')


## Step 4: Build "Without abortion" Tables


In [ ]:
def prepare_without(df, category):
    result = df.head(10).copy()
    result['category'] = category
    result['scenario'] = 'Without abortion'
    result['sex'] = 'Both'
    result['gestation_group'] = None
    result['rank'] = range(1, len(result) + 1)
    result['crude_rate_adjusted'] = result['deaths'] / result['population_adjusted'] * 100_000
    
    # Map to shorter display names for charts
    display_map = {
        'Diseases of heart': 'Heart disease',
        'Malignant neoplasms': 'Cancer',
        'Chronic lower respiratory diseases': 'Respiratory disease',
        'Cerebrovascular diseases': 'Stroke',
        'Alzheimer disease': "Alzheimer's",
        'Diabetes mellitus': 'Diabetes',
        'Accidents (unintentional injuries)': 'Accidents',
        'Intentional self-harm (suicide)': 'Suicide',
        'Chronic liver disease and cirrhosis': 'Liver disease',
        'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
        'Influenza and pneumonia': 'Flu/Pneumonia',
        'Essential hypertension and hypertensive renal disease': 'Hypertension',
        'Assault (homicide)': 'Homicide',
        'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
    }
    result['cause'] = result['cause'].map(display_map).fillna(result['cause'])
    
    return result[['category', 'scenario', 'rank', 'cause_code', 'cause',
                   'deaths', 'sex', 'population', 'population_adjusted',
                   'crude_rate', 'crude_rate_adjusted', 'gestation_group']]

without_national = prepare_without(mort_national, 'National')
without_female = prepare_without(mort_female, 'Female')
without_repro = prepare_without(mort_repro, 'Female 15-44')

print(f'Without tables created:')
print(f'  National: {len(without_national)} rows')
print(f'  Female: {len(without_female)} rows')
print(f'  Female 15-44: {len(without_repro)} rows')


## Step 5: Build "With abortion" Tables

Takes top 10 causes from WITHOUT table, adds abortion row, re-ranks all together.


In [ ]:
def build_with(df_without, abortion_total, category):
    # Start with the WITHOUT table but change scenario to WITH
    df = df_without.copy()
    df['scenario'] = 'With abortion'
    
    # Create abortion row using same population from the first row
    abortion_row = pd.DataFrame([{
        'category': category,
        'scenario': 'With abortion',
        'rank': None,
        'cause_code': 'ABORT',
        'cause': 'Abortion',
        'deaths': int(abortion_total),
        'sex': 'Female',
        'population': df['population'].iloc[0],
        'population_adjusted': df['population_adjusted'].iloc[0],
        'crude_rate': None,
        'crude_rate_adjusted': abortion_total / df['population_adjusted'].iloc[0] * 100_000,
        'gestation_group': '78.6% ≤9w, 14.2% 10-13w, 6.1% 14-20w, 1.1% ≥21w',
    }])
    
    # Combine all rows
    combined = pd.concat([df, abortion_row], ignore_index=True)
    
    # Re-rank by deaths (highest first)
    combined = combined.sort_values('deaths', ascending=False).reset_index(drop=True)
    combined['rank'] = range(1, len(combined) + 1)
    
    return combined

with_national = build_with(without_national, abortion_national, 'National')
with_female = build_with(without_female, abortion_female, 'Female')
with_repro = build_with(without_repro, abortion_repro, 'Female 15-44')

print(f'With tables created:')
print(f'  National: {len(with_national)} rows (abortion rank #{with_national[with_national["cause"] == "Abortion"]["rank"].iloc[0]})')
print(f'  Female: {len(with_female)} rows (abortion rank #{with_female[with_female["cause"] == "Abortion"]["rank"].iloc[0]})')
print(f'  Female 15-44: {len(with_repro)} rows (abortion rank #{with_repro[with_repro["cause"] == "Abortion"]["rank"].iloc[0]})')


## Step 6: Create Master Export Table


In [ ]:
master = pd.concat([without_national, without_female, without_repro,
                    with_national, with_female, with_repro],
                   ignore_index=True)

master['year'] = 2024
master['data_source'] = 'CDC WONDER 2024 + Guttmacher 2024'

# Reorder columns for clarity
master = master[['year', 'category', 'scenario', 'rank', 'cause_code', 'cause',
                 'deaths', 'sex', 'population', 'population_adjusted',
                 'crude_rate', 'crude_rate_adjusted', 'gestation_group', 'data_source']]

print(f'Master table: {len(master)} rows')
print(f'Categories: {master["category"].unique().tolist()}')
print(f'Scenarios: {master["scenario"].unique().tolist()}')
print(f'Sample (first 3 and last 3):')
print(master[['category', 'scenario', 'rank', 'cause', 'deaths']].head(3))
print('...')
print(master[['category', 'scenario', 'rank', 'cause', 'deaths']].tail(3))


## Step 7: Export Files


In [ ]:
export_dir = Path('export')
export_dir.mkdir(exist_ok=True)

# CSV
csv_file = export_dir / 'abortion_cause_of_death_v1.csv'
master.to_csv(csv_file, index=False)
print(f'✓ {csv_file.name}')

# Parquet
parquet_file = export_dir / 'abortion_cause_of_death_v1.parquet'
master.to_parquet(parquet_file, index=False)
print(f'✓ {parquet_file.name}')

# Excel
excel_file = export_dir / 'abortion_cause_of_death_v1.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    for cat in ['National', 'Female', 'Female 15-44']:
        cat_data = master[master['category'] == cat]
        sheet_name = cat.replace(' ', '_')
        cat_data.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'✓ {excel_file.name}')

print(f'\nAll files saved to: {export_dir}')


In [ ]:
con.close()
print('✓ Complete')
